activate environment (i have already created one using command: python -m venv qiime2_work)  
if creating new env make sure to: pip install jupyterlab  

it is recommended by qiime2 devs to create a new env for qiime !  
  
to deactivate: deactivate

In [ ]:
source qiime2_work/bin/activate

#install QIIME below if you haven't already !!
#I installed rachis-qiime2-2026.4

conda env create \
  --name qiime2-2026.4 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.4/qiime2/released/rachis-qiime2-linux-64-conda.yml

QIIME2 manages /artifacts/ which are intermediate data bits during the analysis on which the actions act  
So our first need would be to transform our sequence data into an artifact since QIIME can only do things to those 



In [ ]:
#activate environment
conda activate qiime2-2026.4

#importing SRA files from download (using sra-download.sh bash script) and changing them into a QIIME artifact
#you will need to make a manifest file for your seq data
qiime tools import \
 --type 'SampleData[PairedEndSequencesWithQuality]' \
 --input-path fq-manifest.tsv \
 --output-path demux.qza \
 --input-format PairedEndFastqManifestPhred33V2


#visualize the sequences that you just imported as artifacts 
#this is helpful for choosing the trimming parameters inside of DADA2
  qiime demux summarize \
  --i-data demux.qza \
  --o-visualization demux.qzv

Next, we will use dada2 to create the ASV table, given that we have inspected their sequence quality to find when we should cut off forward and reverse sequences.  
This also assumes that primers have been removed from the sequences as well (this can be checked visually)

In [ ]:

qiime dada2 denoise-paired \
  --i-demultiplexed-seqs demux.qza \
  --p-trim-left-f 0 \
  --p-trunc-len-f 250 \
  --p-trim-left-r 0 \
  --p-trunc-len-r 250 \
  --o-representative-sequences asv-seqs.qza \
  --o-table asv-table.qza \
  --o-denoising-stats denoising-stats.qza \
  --o-base-transition-stats base-transition-stats.qza

If you choose to run DADA2 in R and then import the resultant ASV table into q2 run the following below: 

Following q2 we can run CarveME and create multiple models in parallel: 
see site for more instructions: https://carveme.readthedocs.io/en/latest/usage.html

In [ ]:
carve -r myfolder/*.faa -o mymodels/ 